# Lekcija 12 - Zmanjšanje zgodovine klepeta z agentovim zvezkom

Ta zvezek prikazuje, kako upravljati kontekst v dolgih pogovorih z uporabo Microsoft Agent Framework. Ko pogovori rastejo, se število tokenov povečuje — na koncu preseže kontekstno okno modela. To rešujemo z **vzorec povzetka konteksta** in **agentovim zvezkom** za trajni spomin.

## Kaj se boste naučili:
1. **Zakaj je upravljanje konteksta pomembno**: Razumevanje omejitev tokenov in kontekstnih oken
2. **Agentje z zavedanjem konteksta**: Gradnja agentov, ki upravljajo svoj lasten kontekst pogovora
3. **Vzorec povzetka konteksta**: Uporaba orodij za strnitev zgodovine pogovora
4. **Agentov zvezek**: Trajni spomin, ki preživi zmanjšanje konteksta

## Predpogoji:
- Nastavitev Azure OpenAI z konfiguriranimi okoljskimi spremenljivkami
- Razumevanje osnovnih konceptov agentov iz prejšnjih lekcij


## Namestitev


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Zakaj je upravljanje konteksta pomembno

Vsak LLM ima omejeno **kontekstno okno** — največje število tokenov, ki jih lahko obdela v enem samem zahtevku. Ko poteka večkratni pogovor:

- **Število tokenov linearno narašča** z vsakim sporočilom uporabnika in odgovorom asistenta.
- **Tokeni poziva dominirajo stroške**, ker se celotna zgodovina vsakokrat znova pošlje.
- Sčasoma pogovor **preseže kontekstno okno** in model ali skrajša ali vrže napako.

### Strategije za upravljanje konteksta

| Strategija | Kako deluje | Kompromis |
|---|---|---|
| **Skrajšanje** | Odstrani najstarejša sporočila | Izgubi zgodnji kontekst |
| **Povzemanje** | Stisne starejša sporočila v povzetek | Nekaj podrobnosti izgubljenih, a ključne točke ohranjene |
| **Zvezek / Zunanje pomnjenje** | Shrani ključne informacije izven pogovora | Zahteva klice orodij, a preživi katerokoli zmanjšanje |

V tem zapisku združujemo **povzemanje** z orodjem **zvezka**, da agent ohrani kontinuiteto tudi, ko je zgodovina pogovora stisnjena.


## Ustvarjanje agenta, ki je zavedno o kontekstu


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Simulacija dolgega pogovora

Pojdimo skozi večkrožni pogovor, da vidimo, kako se kontekst kopiči. Agent naj ohrani ključne podrobnosti (preferenc, proračun, datume potovanja) skozi kroge in pokaže kontinuiteto.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Opazite, kako agent ohranja kontekst iz prejšnjih pogovorov — pozna Japonsko, sushi, templje, fotografijo, proračun 3000 $, solo potovanje in april kot časovni okvir. V kratkem pogovoru to dobro deluje, toda ko se pogovor razvija, postane ponovno pošiljanje celotne zgodovine drago.

Nadaljujmo pogovor z več pogovornih zankami, da vidimo kopičenje konteksta:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Vzorec povzemanja konteksta

Ko pogovor napreduje, lahko uporabimo **orodje za povzemanje**, da stisnemo zbran kontekst v jedrnat obliko. Agent pokliče to orodje, da zabeleži ključne preference, tako da tudi če se starejša sporočila izbrišejo, so bistvene informacije ohranjene.

Ta vzorec je gradnik za bolj sofisticirano zmanjševanje zgodovine:
1. Agent identificira ključne dejstva iz pogovora
2. Pokliče orodje za povzemanje, da jih trajno shrani
3. Starejša sporočila se lahko varno odstranijo, ker povzetek zajame, kar je pomembno

Spodaj definiramo orodje `summarize_preferences`, ki ga lahko agent pokliče za zapis jedrnatega povzetka tega, kar se je naučil.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Povzetek

V tej lekciji ste se naučili, kako upravljati s kontekstom v dolgo trajajočih pogovorih z agentom z uporabo Microsoft Agent Framework:

### Ključni pojmi
- **Okna konteksta so omejena** — vsak žeton v zgodovini pogovora stane in se šteje v omejitev.
- **Orodja za povzemanje** agentu omogočajo, da združi zbran kontekst v strnjene povzetke, zmanjšajo uporabo žetonov ob ohranjanju ključnih informacij.
- **Zvezki agenta** zagotavljajo trajni zunanji pomnilnik, ki preživi kakršno koli zmanjšanje pogovora.

### Kaj ste zgradili
- **Agent, ki upošteva kontekst**, ki vzdržuje kontinuiteto v večkrajnih pogovorih
- **Orodje za povzemanje** (`summarize_preferences`), ki v kompaktnem formatu zapisuje ključne uporabniške podatke
- **Večkrajni pogovor**, ki prikazuje ohranjanje konteksta in upravljanje sprememb

### Praktične uporabe
- **Chatboti za storitve za stranke**: Zapomnijo si preference preko dolgih podpornih sej
- **Osebni asistenti**: Spremljajo tekoče projekte brez ponovnega pojasnjevanja konteksta
- **Izobraževalni tutorji**: Ohranjajo napredek študenta skozi več interakcij

### Naslednji koraki
- Uvedite popolno orodje zvezek z vztrajnostjo na osnovi datotek
- Dodajte samodejno krajšanje zgodovine po povzetku
- Združite z vektorskimi bazami za iskanje semantičnega pomnilnika
- Izdelajte agente, ki lahko nadaljujejo pogovore dni pozneje s polnim kontekstom


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
